<h1> Dataset </h1>

In [2]:
from datasets import load_dataset

dataset = load_dataset ("amalia-llm/amalia-PTradutor", "base")["train"] #https://huggingface.co/datasets/amalia-llm/amalia-PTradutor

""" Stats do Dataset """

print (dataset.features)
print (type(dataset)) #<class 'datasets.arrow_dataset.Dataset'> IMPORTANTE ser Dataset e não Columns

print (dataset[0])
print (type(dataset[0]))

ModuleNotFoundError: No module named 'datasets'

<h2> Roles do Dataset </h2>
<hr>

Esta parte é importante para percebermos se o dataset está construido de maneira uniforme com as roles tradicionais.

In [ ]:
roles = set()

for exemplo in dataset["conversations"]:
    for conversas in exemplo:
        roles.add (conversas["from"])
        #print (conversas["from"])

print (roles) #{'user', 'assistant', 'system'}

Eu no meu caso, não quero <i> value </i> quero mudar para <i> content</i>. <br>
Isto é algo comum ao trabalhar com datasets.

In [2]:
def NORMALIZE (data):
    for exemplo in data["conversations"]:
        exemplo["content"] = exemplo.pop ("value") #pop método de lista

    return data

"""
map é uma função da library datasets da Hugging Face que permite aplicar transformações a um dataset 
"""
dataset = dataset.map (NORMALIZE) 

print (dataset[0])

{'conversations': [{'from': 'system', 'content': 'És um tradutor profissional de inglês para português. Traduz aquilo que o utilizador escrever.'}, {'from': 'user', 'content': "Brazil Collor de Mello's fatal week After three months of investigations and revelation of scandals, Brazil anxiously awaits the release of the final report of the Parliamentary Inquiry Commission which, tomorrow, should accuse President Collor de Mello of benefiting from the corruption scheme set up by Paulo César («PC») Farias. Page 10"}, {'from': 'assistant', 'content': 'Brasil A semana fatal de Collor de Mello Após três meses de investigações e revelação de escândalos, o Brasil aguarda ansiosamente a divulgação do relatório final da Comissão de Inquérito Parlamentar que, amanhã, deverá acusar o Presidente Collor de Mello de beneficiar do esquema de corrupção montado por Paulo César («PC») Farias. Pag. 10'}]}


<hr>

<h1> Data Pruning </h1>

Como vamos realizar Data Pruning, vamos precisar de ter os exemplos bem definidos. <br>
Nos datasets de modelos de Linguagem por norma temos sempre as chaves {"system", "user", "assistant"}. <br>
Nós para Data Pruning, não queremos isso, queremos apenas o <b> conteúdo</b> real de cada exemplo.

In [3]:
dataset_noformat = []

for exemplo in dataset["conversations"]:
    """
    Para cada exemplo vamos ficar com as 3 ou 2 chaves e juntar numa variável que depois vai ser adiconada
    à lista.
    """

    content = " ".join (x["content"] for x in exemplo) #content porque mudámos!!

    dataset_noformat.append (content.lower()) #lower para haver uma igualdade de dados

display (type(dataset_noformat))
print (dataset_noformat[4054])
display (dataset_noformat[:5])

list

és um tradutor especializado em inglês-português. a tua tarefa é traduzir o texto do utilizador para português. q - are the neologisms you use in your poetry all your creation, or do you also collect them from the mouths of the people, here in the pantanal? m. b. -- ah yes, i discover many terms in people's speech. what i call plural syntax is practiced by illiterate men, who express themselves in a different way than we do. this syntax of the people, quite plural -- because they are not guided by grammar and therefore can speak in different ways -- i really like that. for me it's the same thing as discovering camilo's syntax, one is erudite, constructed, and the other primary, but each has its own beauty. another thing i like is listening to the children's conversations. i have a poem in my new book where i try to explain children's poetry: «i heard the color of the birds». children practice these types of phrases all the time. a child's language is always a birth. i exploited my five

["és um tradutor profissional de inglês para português. traduz aquilo que o utilizador escrever. brazil collor de mello's fatal week after three months of investigations and revelation of scandals, brazil anxiously awaits the release of the final report of the parliamentary inquiry commission which, tomorrow, should accuse president collor de mello of benefiting from the corruption scheme set up by paulo césar («pc») farias. page 10 brasil a semana fatal de collor de mello após três meses de investigações e revelação de escândalos, o brasil aguarda ansiosamente a divulgação do relatório final da comissão de inquérito parlamentar que, amanhã, deverá acusar o presidente collor de mello de beneficiar do esquema de corrupção montado por paulo césar («pc») farias. pag. 10",
 'por favor, traduz este texto de inglês para português:\nthe state of innocence, symbolized yesterday in the brussels demonstration by the white of the clothes, often does not survive the violence and hypocrisy of the w

<h2> Duplicados usando Hashes </h2>
<hr> 

https://en.wikipedia.org/wiki/Hash_function <br>
Um Hash é uma representação de algo a partir de uma assinatura única. <br>
É perfeito para duplicados porque cada exemplo vai ter uma assinatura única mas se houver exemplos iguais, ambos vão ter assinaturas iguais. <br>
É uma maneira rápida e barata de remover duplicados. <br>
Não compensa comparar exemplo a exemplo porque é muito mais caro!

In [ ]:
import hashlib #https://docs.python.org/3/library/hashlib.html

"""
Uma maneira muito boa de ver os duplicados é usar set.
Set é uma maneira de armazenar dados em Python que não permite duplicados.
Dito isto, se houver alguma assinatura duplicada, não irá ser adicionada ao set.
"""
DUPLICADOS = set()
HASH = [] #Para vermos realmente o que o algoritmo Hash faz

for exemplo in dataset_noformat:

    hs = hashlib.blake2b (exemplo.encode()).hexdigest()

    HASH.append (hs)
    DUPLICADOS.add (hs)


print (len(HASH)) #Aqui tem que obrigatoriamente que dar o número de exemplos do dataset

"""
Se len(duplicados) == len(dataset) é porque NÃO há exemplos duplicados.
Caso contrário, há duplicados.
"""
print (len(DUPLICADOS))
print (f"Número de Duplicados: {len(HASH) - len(DUPLICADOS)}")

<h2> Duplicados Lexicais </h2>
<hr>

https://www.pinecone.io/learn/series/faiss/locality-sensitive-hashing/ <br>
Os Duplicados Lexicais representam <i> near-duplicates</i> a nivel Lexical. <br>
Desta maneira conseguimos perceber se um exemplo é parecido a outro a nivel Lexical. <br>

Para esta etapa usamos uma técnica parecida ao Hash.. MinHash. <br>
<b>MinHash</b> aplica um Hash a cada palavra ou <b> shingle</b>. <br>
A ideia é depois comparar todos esses Hashes exemplo a exemplo mas isto é muito caro. <br>

Por isso usamos <b>Locality Sensitive Hashing</b> que utiliza partes dessas assinaturas para colocar exemplos parecidos num grupo e assim só compara todas as assinaturas com esses exemplos.

Esta técnica usa Similaridade de Jaccard para comparar exemplos.

In [ ]:
from datasketch import MinHash
##### Exemplo para Perceber

MinHash_1 = MinHash ()
MinHash_2 = MinHash ()

#Exemplo 1
for palavra in dataset_noformat[0].split ():
    MinHash_1.update (palavra.encode ("utf-8"))

#Exemplo 2
for palavra in dataset_noformat[1].split ():
    MinHash_2.update (palavra.encode ("utf-8"))


print (f"Similaridade de Jaccard entre o exemplo 0 e o exemplo 1 do dataset: {MinHash_1.jaccard (MinHash_2) * 100}%")


"""
Agora imaginemos isto a uma escala de centenas de milhares de exemplos!!
É aqui entra Locality Sensitive Hashing.
"""

In [ ]:
"""
Para similaridade lexical, usar apenas palavras não acaba por ser a melhor solução.
Estamos apenas a comparar palavra a palavra sem ter todo o contexto por trás que acaba
por ser relevante para a compreensão lexical.

Por esse motivo usamos Shingles, que são nada mais nada menos do que n-grams.
    Exemplo:

        {"o", "gato", "comeu", "o", "rato"}   |    {"o", "gato", "não", "comeu" "o" "rato"}    
    
Se comparámos palavra a palavra claramente ambas as frases são parecidas mas qualquer humano consegue
perceber que não tem o mesmo valor lexical.
"""

def SHINGLES (texto, k = 5):

    palavras = texto.split()

    return [" ".join(palavras [i : i + k]) for i in range (len (palavras) - k + 1)]

x = SHINGLES (dataset_noformat[0])
print (x)
print (len(x))

In [ ]:
from datasketch import MinHashLSH #https://github.com/ekzhu/datasketch

"""
Aqui usamos LSH para o mesmo meter em grupos os exemplos candidatos a serem parecidos.
Ainda não é Similaridade Jaccard!!
"""

MinimunHashLSH = {}

LocalitySensitiveHashing = MinHashLSH (threshold = 0.55)
#threshold define o indice de Similaridade de Jaccard, se um documento obter mais de 0.8, é semelhante 

for idx, exemplo in enumerate (dataset_noformat):

    MinimunHash = MinHash () #Para cada exemplo criamos um MinHash novo

    for ngrams in SHINGLES (exemplo):

        MinimunHash.update (ngrams.encode ("utf-8")) #MinHash de cada Shingle

    MinimunHashLSH [f"Exemplo {idx}"] = MinimunHash #Guardamos num dict
    LocalitySensitiveHashing.insert (f"Exemplo {idx}", MinimunHash) #Guardamos no idx do LSH

### 3min
print (MinimunHashLSH["Exemplo 223"])

In [ ]:
"""
Agora com o dict e com o indice LSH podemos percorrer cada exemplo e encontrar os candidatos a mais semelhantes.
É preciso ter atenção que isto ainda não são os mais semelhantes, são os CANDIDATOS a mais semelhantes.
"""

Candidatos = []

for nome, minash in MinimunHashLSH.items():

    #print (f"{nome} -> {minash}")

    LSH = LocalitySensitiveHashing.query (minash)

    Candidatos.append (LSH)


"""
LSH devolve uma lista com o exemplo que pedimos para procurar os candidatos mais parecidos e os 
candidatos mais parecidos caso exista. 
Nesse sentido podemos procurar os candidatos a dups procurando pela len.
"""

SimilJaccard_Set = set()

for x in Candidatos:

    dups = len(x)

    if dups >= 2:

        MinimunHash_1 = MinimunHashLSH[x[0]]
        MinimunHash_2 = MinimunHashLSH[x[1]]

        SimilJaccard = MinimunHash_1.jaccard (MinimunHash_2)

        SimilJaccard_Set.add (f"{x}: {SimilJaccard * 100}%")

print (SimilJaccard_Set) #Se reparámos há exemplos que têm similaridade relativamente baixa, o que comprova que LSH fornece apenas os candidatos e não os mais semelhantes de todo

<hr>
<hr>

<h1> Heurísticas </h1>

<h3> Length Profiler </h3>

In [4]:
heu = {}

for idx, exemplo in enumerate (dataset_noformat):

    chars = ''.join (exemplo.split())
    #print (exemplo)
    #print (len(chars))

    words = exemplo.split()
    words_unique = set (words)

    heu[f"Exemplo {idx}"] = {
        "Número de Chars": len(chars),
        
        "Número de Words": len(words),
        "Número de Words Únicas": len(words_unique),

        "Chars por Words": len(chars) / len(words),
        "Diversidade Lexical": len(words_unique) / len(words) * 100
        }

print (heu["Exemplo 34444"])
print (heu["Exemplo 322"])
print (heu["Exemplo 1093"])

print (x for x in heu.items())

{'Número de Chars': 1867, 'Número de Words': 369, 'Número de Words Únicas': 238, 'Chars por Words': 5.059620596205962, 'Diversidade Lexical': 64.49864498644986}
{'Número de Chars': 2355, 'Número de Words': 493, 'Número de Words Únicas': 302, 'Chars por Words': 4.776876267748479, 'Diversidade Lexical': 61.257606490872206}
{'Número de Chars': 2294, 'Número de Words': 473, 'Número de Words Únicas': 301, 'Chars por Words': 4.849894291754757, 'Diversidade Lexical': 63.63636363636363}
<generator object <genexpr> at 0x000001814D1A06C0>


In [ ]:
import numpy as np

chars = []
words = []

for x in heu:

    chars.append(heu[x]["Número de Chars"])
    words.append (heu[x]["Número de Words"])


chars_P1 = np.percentile (chars, 1)
chars_P5 = np.percentile (chars, 5)
chars_P25 = np.percentile (chars, 25)
chars_P50 = np.percentile (chars, 50)
chars_P75 = np.percentile (chars, 75)
chars_P90 = np.percentile (chars, 90)
chars_P95 = np.percentile (chars, 95)
chars_P99 = np.percentile (chars, 99)

words_P1 = np.percentile (words, 1)
words_P5 = np.percentile (words, 5)
words_P25 = np.percentile (words, 25)
words_P50 = np.percentile (words, 50)
words_P75 = np.percentile (words, 75)
words_P90 = np.percentile (words, 90)
words_P95 = np.percentile (words, 95)
words_P99 = np.percentile (words, 99)


print ("Avaliação Número de Chars:")
print ("P1", chars_P1) #1% dos Exemplos têm < 397 chars
print ("P5", chars_P5) #5% dos Exemplos têm < 459 chars
print ("P25", chars_P25) #25% dos Exemplos têm < 935 chars
print ("P50", chars_P50) #50% dos Exemplos têm < 2173 chars #Mediana
print ("P75", chars_P75) #75% dos Exemplos têm < 2435 chars
print ("P90", chars_P90) #90% dos Exemplos têm < 2525 chars
print ("P95", chars_P95) #95% dos Exemplos têm < 2567 chars
print ("P99", chars_P99) #99% dos Exemplos têm < 2768 chars

print ("\nAvaliação Número de Palavras:")
print ("P1", words_P1)
print ("P5", words_P5)
print ("P25", words_P25)
print ("P50", words_P25)
print ("P75", words_P75)
print ("P90", words_P90)
print ("P95", words_P95)
print ("P99", words_P99)


for idx, values in heu.items():
    
#######################################################
                    #CHARS
#######################################################

    chars = values["Número de Chars"]

    if chars <= chars_P1:

        values["Grupo Percentil Chars"] = "<P1"

    elif chars_P1 < chars <= chars_P5:

        values["Grupo Percentil Chars"] = "P1 - P5"

    elif chars_P5 < chars <= chars_P25:

        values["Grupo Percentil Chars"] = "P5 - P25"

    elif chars_P25 < chars <= chars_P50:

        values["Grupo Percentil Chars"] = "P25 - P50"

    elif chars_P50 < chars <= chars_P75:

        values["Grupo Percentil Chars"] = "P50 - P75"

    elif chars_P75 <chars <= chars_P90:

        values["Grupo Percentil Chars"] = "P75 - P90"

    elif chars_P90 < chars <= chars_P95:

        values["Grupo Percentil Chars"] = "P90 - P95"

    elif chars_P95 < chars <= chars_P99:

        values["Grupo Percentil Chars"] = "P95 - P99"

    else:

        values["Grupo Percentil Chars"] = ">P99"

#######################################################
                    #WORDS
#######################################################

    words = values["Número de Words"]

    if words <= words_P1:

        values["Grupo Percentil Words"] = "<P1"

    elif words_P1 < words <= words_P5:

        values["Grupo Percentil Words"] = "P1 - P5"

    elif words_P5 < words <= words_P25:

        values["Grupo Percentil Words"] = "P5 - P25"

    elif words_P25 < words <= words_P50:

        values["Grupo Percentil Words"] = "P25 - P50"

    elif words_P50 < words <= words_P75:

        values["Grupo Percentil Words"] = "P50 - P75"

    elif words_P75 < words <= words_P90:

        values["Grupo Percentil Words"] = "P75 - P90"

    elif words_P90 < words <= words_P95:

        values["Grupo Percentil Words"] = "P90 - P95"

    elif words_P95 < words <= words_P99:

        values["Grupo Percentil Words"] = "P95 - P99"

    else:

        values["Grupo Percentil Words"] = ">P99"


display (heu)

Avaliação Número de Chars:
P1 397.0
P5 459.0
P25 935.0
P50 2173.0
P75 2435.0
P90 2525.0
P95 2567.0
P99 2768.0

Avaliação Número de Palavras:
P1 78.0
P5 92.0
P25 183.0
P50 183.0
P75 485.0
P90 506.0
P95 520.0
P99 559.0


{'Exemplo 0': {'Número de Chars': 656,
  'Número de Words': 120,
  'Número de Words Únicas': 85,
  'Chars por Words': 5.466666666666667,
  'Diversidade Lexical': 70.83333333333334,
  'Grupo Percentil Chars': 'P25',
  'Grupo Percentil Words': 'P5 - P25'},
 'Exemplo 1': {'Número de Chars': 2514,
  'Número de Words': 483,
  'Número de Words Únicas': 317,
  'Chars por Words': 5.204968944099379,
  'Diversidade Lexical': 65.63146997929607,
  'Grupo Percentil Chars': 'P90',
  'Grupo Percentil Words': 'P50 - P75'},
 'Exemplo 2': {'Número de Chars': 2869,
  'Número de Words': 593,
  'Número de Words Únicas': 348,
  'Chars por Words': 4.838111298482294,
  'Diversidade Lexical': 58.68465430016864,
  'Grupo Percentil Chars': '>P99',
  'Grupo Percentil Words': '>P99'},
 'Exemplo 3': {'Número de Chars': 2445,
  'Número de Words': 498,
  'Número de Words Únicas': 316,
  'Chars por Words': 4.909638554216867,
  'Diversidade Lexical': 63.45381526104418,
  'Grupo Percentil Chars': 'P90',
  'Grupo Percent

<h3> Repetition Profiler </h3>

In [7]:
#### Words Repetidas

for x, values in heu.items():

    values["Número de Words Repetidas"] = values["Número de Words"] - values["Número de Words Únicas"]
    values["% de Número de Words Repetidas"] = values["Número de Words Repetidas"] / values["Número de Words"] * 100

display (heu["Exemplo 0"])


#### N Grams Repetidos

def NGRAMS (texto, k):

    palavras = texto.split()

    return [" ".join(palavras [i : i + k]) for i in range (len (palavras) - k + 1)]

for idx, value in heu.items():

    exemplo = dataset_noformat [int(idx.split()[-1])] #Genialidade 

    """
    Bi Gram
    """

    n = NGRAMS (exemplo, k = 2)

    ngrams = []
    nodups = set()

    for x in n:

        ngrams.append (x)
        nodups.add (x)

    value["Número de Bi-Grams"] = len(ngrams)
    value["Número de Bi-Grams Repetidos"] = len(ngrams) - len(nodups)
    value["% de Bi-Grams Repetidos"] = (1 - (len(nodups) / len(ngrams))) * 100

    """
    Tri Gram
    """

    n = NGRAMS (exemplo, k = 3)

    ngrams = []
    nodups = set()

    for x in n:

        ngrams.append (x)
        nodups.add (x)
        
    value["Número de Tri-Grams"] = len(ngrams)
    value["Número de Tri-Grams Repetidos"] = len(ngrams) - len(nodups)
    value["% de Tri-Grams Repetidos"] = (1 - (len(nodups) / len(ngrams))) * 100


display (heu["Exemplo 29334"])

{'Número de Chars': 656,
 'Número de Words': 120,
 'Número de Words Únicas': 85,
 'Chars por Words': 5.466666666666667,
 'Diversidade Lexical': 70.83333333333334,
 'Grupo Percentil Chars': 'P25',
 'Grupo Percentil Words': 'P5 - P25',
 'Número de Words Repetidas': 35,
 '% de Número de Words Repetidas': 29.166666666666668,
 'Número de Bi-Grams': 119,
 'Número de Bi-Grams Repetidos': 9,
 '% de Bi-Grams Repetidos': 7.563025210084029,
 'Número de Tri-Grams': 118,
 'Número de Tri-Grams Repetidos': 4,
 '% de Tri-Grams Repetidos': 3.3898305084745783}

{'Número de Chars': 620,
 'Número de Words': 121,
 'Número de Words Únicas': 90,
 'Chars por Words': 5.12396694214876,
 'Diversidade Lexical': 74.3801652892562,
 'Grupo Percentil Chars': 'P25',
 'Grupo Percentil Words': 'P5 - P25',
 'Número de Words Repetidas': 31,
 '% de Número de Words Repetidas': 25.6198347107438,
 'Número de Bi-Grams': 120,
 'Número de Bi-Grams Repetidos': 6,
 '% de Bi-Grams Repetidos': 5.000000000000004,
 'Número de Tri-Grams': 119,
 'Número de Tri-Grams Repetidos': 2,
 '% de Tri-Grams Repetidos': 1.6806722689075682}

In [11]:
words_ratio = []
bigram_ratio = []
trigram_ratio = []

for idx, value in heu.items():

    words_ratio.append (value["% de Número de Words Repetidas"])
    bigram_ratio.append (value["% de Bi-Grams Repetidos"])
    trigram_ratio.append (value["% de Tri-Grams Repetidos"])



#### Words Descriminação
words_ratio_P1 = np.percentile (words_ratio, 1)
words_ratio_P5 = np.percentile (words_ratio, 5)
words_ratio_P25 = np.percentile (words_ratio, 25)
words_ratio_P50 = np.percentile (words_ratio, 50)
words_ratio_P75 = np.percentile (words_ratio, 75)
words_ratio_P90 = np.percentile (words_ratio, 90)
words_ratio_P95 = np.percentile (words_ratio, 95)
words_ratio_P99 = np.percentile (words_ratio, 99)
print (f"Words Stats: \nP1: {words_ratio_P1}\nP5: {words_ratio_P5}\nP25: {words_ratio_P25}\nP50: {words_ratio_P50}\nP75: {words_ratio_P75}\nP90: {words_ratio_P90}\nP95: {words_ratio_P95}\nP99: {words_ratio_P99}")

#### Bi Gram Descriminação
bigram_ratio_P1 = np.percentile (bigram_ratio, 1)
bigram_ratio_P5 = np.percentile (bigram_ratio, 5)
bigram_ratio_P25 = np.percentile (bigram_ratio, 25)
bigram_ratio_P50 = np.percentile (bigram_ratio, 50)
bigram_ratio_P75 = np.percentile (bigram_ratio, 75)
bigram_ratio_P90 = np.percentile (bigram_ratio, 90)
bigram_ratio_P95 = np.percentile (bigram_ratio, 95)
bigram_ratio_P99 = np.percentile (bigram_ratio, 99)
print (f"\nBi Gram Stats: \nP1: {bigram_ratio_P1}\nP5: {bigram_ratio_P5}\nP25: {bigram_ratio_P25}\nP50: {bigram_ratio_P50}\nP75: {bigram_ratio_P75}\nP90: {bigram_ratio_P90}\nP95: {bigram_ratio_P95}\nP99: {bigram_ratio_P99}")

#### Tri Gram Descriminação
trigram_ratio_P1 = np.percentile (trigram_ratio, 1)
trigram_ratio_P5 = np.percentile (trigram_ratio, 5)
trigram_ratio_P25 = np.percentile (trigram_ratio, 25)
trigram_ratio_P50 = np.percentile (trigram_ratio, 50)
trigram_ratio_P75 = np.percentile (trigram_ratio, 75)
trigram_ratio_P90 = np.percentile (trigram_ratio, 90)
trigram_ratio_P95 = np.percentile (trigram_ratio, 95)
trigram_ratio_P99 = np.percentile (trigram_ratio, 99)
print (f"\nTri Gram Stats: \nP1: {trigram_ratio_P1}\nP5: {trigram_ratio_P5}\nP25: {trigram_ratio_P25}\nP50 {trigram_ratio_P50}\nP75: {trigram_ratio_P75}\nP90: {trigram_ratio_P90}\nP95: {trigram_ratio_P95}\nP99: {trigram_ratio_P99}")

"""
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
"""

for idx, values in heu.items():
    
#######################################################
                    #WORDS
#######################################################

    words_ratio = values["% de Número de Words Repetidas"]

    if words_ratio <= words_ratio_P1:

        values["Grupo Percentil % de Número de Words Repetidas"] = "<P1"

    elif words_ratio_P1 < words_ratio <= words_ratio_P5:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P1 - P5"

    elif words_ratio_P5 < words_ratio <= words_ratio_P25:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P5 - P25"

    elif words_ratio_P25 < words_ratio <= words_ratio_P50:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P25 - P50"

    elif words_ratio_P50 < words_ratio <= words_ratio_P75:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P50 - P75"

    elif words_ratio_P75 < words_ratio <= words_ratio_P90:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P75 - P90"

    elif words_ratio_P90 < words_ratio <= words_ratio_P95:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P90 - P95"

    elif words_ratio_P95 < words_ratio <= words_ratio_P99:

        values["Grupo Percentil % de Número de Words Repetidas"] = "P95 - P99"

    else:

        values["Grupo Percentil % de Número de Words Repetidas"] = ">P99"

#######################################################
                    #BI-GRAMS
#######################################################

    bigram_ratio = values["% de Bi-Grams Repetidos"]

    if bigram_ratio <= bigram_ratio_P1:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "<P1"

    elif bigram_ratio_P1 < bigram_ratio <= bigram_ratio_P5:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P1 - P5"

    elif bigram_ratio_P5 < bigram_ratio <= bigram_ratio_P25:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P5 - P25"

    elif bigram_ratio_P25 < bigram_ratio <= bigram_ratio_P50:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P25 - P50"

    elif bigram_ratio_P50 < bigram_ratio <= bigram_ratio_P75:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P50 - P75"

    elif bigram_ratio_P75 < bigram_ratio <= bigram_ratio_P90:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P75 - P90"

    elif bigram_ratio_P90 < bigram_ratio <= bigram_ratio_P95:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P90 - P95"

    elif bigram_ratio_P95 < bigram_ratio <= bigram_ratio_P99:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = "P95 - P99"

    else:

        values["Grupo Percentil % de Bi-Grams Repetidos"] = ">P99"

#######################################################
                    #TRI-GRAMS
#######################################################

    trigram_ratio = values["% de Tri-Grams Repetidos"]

    if trigram_ratio <= trigram_ratio_P1:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "<P1"

    elif trigram_ratio_P1 < trigram_ratio <= trigram_ratio_P5:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P1 - P5"

    elif trigram_ratio_P5 < trigram_ratio <= trigram_ratio_P25:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P5 - P25"

    elif trigram_ratio_P25 < trigram_ratio <= trigram_ratio_P50:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P25 - P50"

    elif trigram_ratio_P50 < trigram_ratio <= trigram_ratio_P75:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P50 - P75"

    elif trigram_ratio_P75 < trigram_ratio <= trigram_ratio_P90:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P75 - P90"

    elif trigram_ratio_P90 < trigram_ratio <= trigram_ratio_P95:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P90 - P95"

    elif trigram_ratio_P95 < trigram_ratio <= trigram_ratio_P99:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = "P95 - P99"

    else:

        values["Grupo Percentil % de Tri-Grams Repetidos"] = ">P99"

display (heu["Exemplo 5454"])

Words Stats: 
P1: 12.871287128712872
P5: 17.97752808988764
P25: 28.205128205128204
P50: 35.051546391752574
P75: 38.7308533916849
P90: 41.61616161616162
P95: 43.51464435146444
P99: 47.950881367882964

Bi Gram Stats: 
P1: 0.0
P5: 0.5680213148433542
P25: 2.7272727272727226
P50: 4.402515723270439
P75: 6.33187772925764
P90: 8.530899721658015
P95: 10.309278350515461
P99: 15.277777777777779

Tri Gram Stats: 
P1: 0.0
P5: 0.0
P25: 0.0
P50 0.6993006993006978
P75: 1.6326530612244872
P90: 2.985074626865669
P95: 4.262911864733502
P99: 8.25688073394495


{'Número de Chars': 1440,
 'Número de Words': 266,
 'Número de Words Únicas': 184,
 'Chars por Words': 5.413533834586466,
 'Diversidade Lexical': 69.17293233082707,
 'Grupo Percentil Chars': 'P50',
 'Grupo Percentil Words': 'P25 - P50',
 'Número de Words Repetidas': 82,
 '% de Número de Words Repetidas': 30.82706766917293,
 'Número de Bi-Grams': 265,
 'Número de Bi-Grams Repetidos': 10,
 '% de Bi-Grams Repetidos': 3.7735849056603765,
 'Número de Tri-Grams': 264,
 'Número de Tri-Grams Repetidos': 0,
 '% de Tri-Grams Repetidos': 0.0,
 'Grupo Percentil % de Número de Words Repetidas': 'P25 - P50',
 'Grupo Percentil % de Bi-Grams Repetidos': 'P25 - P50',
 'Grupo Percentil % de Tri-Grams Repetidos': '<P1'}